# Clases y objetos · TAD de conjuntos

Un conjunto es una colección en la que no importan ni el orden ni las repeticiones. Como TAD se define con seis operaciones primitivas (`vacio`, `insertar`, `eliminar`, `pertenece`, `es_vacio` y `menor`) y las propiedades que deben cumplir. Aquí se implementa de cuatro formas intercambiables (lista no ordenada con duplicados, lista no ordenada sin duplicados, lista ordenada sin duplicados con búsqueda binaria y `frozenset`), se compara el coste de cada operación y se construye sobre la interfaz, sin mirar nunca la representación, el álgebra de conjuntos: subconjuntos, unión, intersección, diferencias, particiones, imagen y producto cartesiano. Las implementaciones son persistentes (cada operación devuelve un conjunto nuevo) y todo se comprueba con [Hypothesis](https://hypothesis.readthedocs.io/) contra el `set` de Python, con un perfil reproducible (`derandomize=True`, sin base de datos de fallos).

Continúa *TAD de pilas y colas*. Ideas del capítulo 9 del libro [*Ejercicios de programación con Python*](https://github.com/jaalonso/Ejercicios-Python) de José A. Alonso Jiménez.

In [1]:
import bisect
import math
import timeit
from abc import ABC, abstractmethod
from functools import reduce
from itertools import product as producto_itertools
from itertools import zip_longest
from typing import Self

from hypothesis import given, settings
from hypothesis import strategies as st

settings.register_profile("cuaderno", derandomize=True, database=None, deadline=None)
settings.load_profile("cuaderno")


def comprobar(prueba, nombre=None):
    """Ejecuta una prueba de Hypothesis; si falla, muestra el contraejemplo reducido y devuelve el error."""
    nombre = nombre or prueba.__name__
    try:
        prueba()
    except Exception as e:
        print(f"{nombre}: FALLA ({type(e).__name__})")
        for nota in getattr(e, "__notes__", []):
            # La explicación que sigue incluye rutas de ficheros locales
            if nota.startswith("Explanation"):
                break
            print("   ", nota.replace("\n", "\n    "))
        return e
    print(f"{nombre}: pasa")

## 1. La interfaz del TAD conjunto

Definir la clase abstracta genérica `Conjunto[T]` (con elementos comparables) y sus operaciones primitivas: `vacio()` (método de clase), `insertar(x)`, `eliminar(x)`, `pertenece(x)`, `es_vacio()` y `menor()`, que lanza `ValueError` en el conjunto vacío. Escribir el generador `elementos(c)`, que dé los elementos de menor a mayor usando solo `menor` y `eliminar`, y con él `__str__` (`{2, 3, 5}` o `∅`), `__repr__` y `__eq__`. Toda implementación debe cumplir, para cualesquiera $x$, $y$ y $c$:

- $\text{insertar}(x, \text{insertar}(x, c)) = \text{insertar}(x, c)$ y $\text{insertar}(x, \text{insertar}(y, c)) = \text{insertar}(y, \text{insertar}(x, c))$.
- $x \notin \text{vacio}$ e $y \in \text{insertar}(x, c) \iff y = x \lor y \in c$.
- $\text{eliminar}(x, \text{vacio}) = \text{vacio}$; $\text{eliminar}(x, \text{insertar}(x, c)) = \text{eliminar}(x, c)$ y, si $x \ne y$, $\text{eliminar}(x, \text{insertar}(y, c)) = \text{insertar}(y, \text{eliminar}(x, c))$.
- $\text{vacio}$ es vacío e $\text{insertar}(x, c)$ no lo es; si $c$ no es vacío, $\text{menor}(c) \in c$ y no hay ningún elemento menor.

In [2]:
def elementos(c):
    while not c.es_vacio():
        m = c.menor()
        yield m
        c = c.eliminar(m)


_FALTA = object()


class Conjunto[T](ABC):
    """TAD conjunto: solo los métodos abstractos conocen la representación."""

    __slots__ = ()

    @classmethod
    @abstractmethod
    def vacio(cls) -> Self: ...

    @abstractmethod
    def insertar(self, x: T) -> Self: ...

    @abstractmethod
    def eliminar(self, x: T) -> Self: ...

    @abstractmethod
    def pertenece(self, x: T) -> bool: ...

    @abstractmethod
    def es_vacio(self) -> bool: ...

    @abstractmethod
    def menor(self) -> T: ...

    def __str__(self):
        return "{" + ", ".join(map(str, elementos(self))) + "}" if not self.es_vacio() else "∅"

    def __repr__(self):
        return f"{type(self).__name__}({self})"

    def __eq__(self, otro):
        if not isinstance(otro, Conjunto):
            return NotImplemented
        return all(a == b for a, b in zip_longest(elementos(self), elementos(otro), fillvalue=_FALTA))

## 2. Listas no ordenadas, con y sin duplicados

Implementar `ConjuntoConDuplicados`, que añade cada elemento al final de una lista aunque ya esté (insertar no busca, pero las demás operaciones recorren todas las copias) y `ConjuntoSinDuplicados`, que solo lo añade si no está. Para que sean persistentes, cada operación trabaja sobre una copia de la lista. Insertar 5, 2, 3, 5, 2 y 4 en las dos, mostrar la lista interna de cada una y comprobar que, aun así, son el mismo conjunto.

In [3]:
class ConjuntoConDuplicados[T](Conjunto[T]):
    __slots__ = ("_elementos",)

    def __init__(self, elementos=()):
        self._elementos = list(elementos)

    @classmethod
    def vacio(cls):
        return cls()

    def insertar(self, x):
        return type(self)(self._elementos + [x])

    def eliminar(self, x):
        return type(self)(y for y in self._elementos if y != x)

    def pertenece(self, x):
        return x in self._elementos

    def es_vacio(self):
        return not self._elementos

    def menor(self):
        if not self._elementos:
            raise ValueError("menor de un conjunto vacío")
        return min(self._elementos)


class ConjuntoSinDuplicados[T](Conjunto[T]):
    __slots__ = ("_elementos",)

    def __init__(self, elementos=()):
        self._elementos = list(elementos)

    @classmethod
    def vacio(cls):
        return cls()

    def insertar(self, x):
        return self if x in self._elementos else type(self)(self._elementos + [x])

    def eliminar(self, x):
        if x not in self._elementos:
            return self
        xs = self._elementos.copy()
        xs.remove(x)
        return type(self)(xs)

    def pertenece(self, x):
        return x in self._elementos

    def es_vacio(self):
        return not self._elementos

    def menor(self):
        if not self._elementos:
            raise ValueError("menor de un conjunto vacío")
        return min(self._elementos)


datos = [5, 2, 3, 5, 2, 4]
c1 = reduce(lambda c, x: c.insertar(x), datos, ConjuntoConDuplicados.vacio())
c2 = reduce(lambda c, x: c.insertar(x), datos, ConjuntoSinDuplicados.vacio())
for c in (c1, c2):
    print(f"{type(c).__name__:22} interna: {str(c._elementos):20} conjunto: {c}")
print("¿Iguales?", c1 == c2, "  menor:", c1.menor(), "  sin el 5:", c1.eliminar(5), "  c1 sigue:", c1)
assert c1 == c2 and c1.eliminar(5) == c2.eliminar(5) and c1.pertenece(4) and not c2.pertenece(7)

ConjuntoConDuplicados  interna: [5, 2, 3, 5, 2, 4]   conjunto: {2, 3, 4, 5}
ConjuntoSinDuplicados  interna: [5, 2, 3, 4]         conjunto: {2, 3, 4, 5}
¿Iguales? True   menor: 2   sin el 5: {2, 3, 4}   c1 sigue: {2, 3, 4, 5}


## 3. Lista ordenada con búsqueda binaria y `frozenset`

Implementar `ConjuntoOrdenado`, sobre una lista ordenada sin repetidos en la que `pertenece`, `insertar` y `eliminar` localizan la posición con `bisect.bisect_left` y el menor es el primer elemento, y `ConjuntoFrozenset`, que delega en un `frozenset` (que ya es inmutable). Repetir la prueba anterior con las cuatro implementaciones y mostrar qué pasa al pedir el menor de un conjunto vacío.

In [4]:
class ConjuntoOrdenado[T](Conjunto[T]):
    __slots__ = ("_elementos",)

    def __init__(self, elementos=()):
        self._elementos = list(elementos)

    @classmethod
    def vacio(cls):
        return cls()

    def _posicion(self, x):
        i = bisect.bisect_left(self._elementos, x)
        return i, i < len(self._elementos) and self._elementos[i] == x

    def insertar(self, x):
        i, esta = self._posicion(x)
        return self if esta else type(self)(self._elementos[:i] + [x] + self._elementos[i:])

    def eliminar(self, x):
        i, esta = self._posicion(x)
        return type(self)(self._elementos[:i] + self._elementos[i + 1 :]) if esta else self

    def pertenece(self, x):
        return self._posicion(x)[1]

    def es_vacio(self):
        return not self._elementos

    def menor(self):
        if not self._elementos:
            raise ValueError("menor de un conjunto vacío")
        return self._elementos[0]


class ConjuntoFrozenset[T](Conjunto[T]):
    __slots__ = ("_elementos",)

    def __init__(self, elementos=frozenset()):
        self._elementos = frozenset(elementos)

    @classmethod
    def vacio(cls):
        return cls()

    def insertar(self, x):
        return type(self)(self._elementos | {x})

    def eliminar(self, x):
        return type(self)(self._elementos - {x})

    def pertenece(self, x):
        return x in self._elementos

    def es_vacio(self):
        return not self._elementos

    def menor(self):
        if not self._elementos:
            raise ValueError("menor de un conjunto vacío")
        return min(self._elementos)


IMPLEMENTACIONES = (ConjuntoConDuplicados, ConjuntoSinDuplicados, ConjuntoOrdenado, ConjuntoFrozenset)

for impl in IMPLEMENTACIONES:
    c = reduce(lambda c, x: c.insertar(x), datos, impl.vacio())
    print(f"{impl.__name__:22} interna: {str(c._elementos):26} conjunto: {c}")
    assert c == c1
try:
    ConjuntoOrdenado.vacio().menor()
except ValueError as e:
    print("ValueError:", e)

ConjuntoConDuplicados  interna: [5, 2, 3, 5, 2, 4]         conjunto: {2, 3, 4, 5}
ConjuntoSinDuplicados  interna: [5, 2, 3, 4]               conjunto: {2, 3, 4, 5}
ConjuntoOrdenado       interna: [2, 3, 4, 5]               conjunto: {2, 3, 4, 5}
ConjuntoFrozenset      interna: frozenset({2, 3, 4, 5})    conjunto: {2, 3, 4, 5}
ValueError: menor de un conjunto vacío


## 4. Las propiedades del TAD

Escribir la estrategia `conjuntos(impl)`, que genere conjuntos de una implementación con una secuencia aleatoria de inserciones (con repeticiones) y eliminaciones, y comprobar con Hypothesis que las cuatro implementaciones cumplen todas las propiedades del ejercicio 1.

In [5]:
def conjuntos(impl, valores=st.integers(0, 15), max_size=12):
    """Conjuntos construidos con insertar (True, x) y eliminar (False, x) mezclados."""

    def construir(operaciones):
        c = impl.vacio()
        for inserta, x in operaciones:
            c = c.insertar(x) if inserta else c.eliminar(x)
        return c

    return st.lists(st.tuples(st.booleans() | st.just(True), valores), max_size=max_size).map(construir)


for impl in IMPLEMENTACIONES:

    @given(conjuntos(impl), st.integers(0, 15), st.integers(0, 15))
    def axiomas(c, x, y):
        v = impl.vacio()
        assert c.insertar(x).insertar(x) == c.insertar(x)
        assert c.insertar(y).insertar(x) == c.insertar(x).insertar(y)
        assert not v.pertenece(x)
        assert c.insertar(x).pertenece(y) == (x == y or c.pertenece(y))
        assert v.eliminar(x) == v
        if x == y:
            assert c.insertar(y).eliminar(x) == c.eliminar(x)
        else:
            assert c.insertar(y).eliminar(x) == c.eliminar(x).insertar(y)
        assert v.es_vacio() and not c.insertar(x).es_vacio()
        if not c.es_vacio():
            m = c.menor()
            assert c.pertenece(m) and not any(c.pertenece(z) for z in range(m))

    assert comprobar(axiomas, f"axiomas de {impl.__name__}") is None

axiomas de ConjuntoConDuplicados: pasa


axiomas de ConjuntoSinDuplicados: pasa


axiomas de ConjuntoOrdenado: pasa
axiomas de ConjuntoFrozenset: pasa


## 5. Coste de cada operación

Mostrar una tabla con el coste de cada operación en cada implementación, donde $n$ es el número de elementos distintos y $m \ge n$ el de elementos guardados con repeticiones. Después comprobarla midiendo, para conjuntos de $n = 1000$ y $n = 32\,000$ elementos, cuánto crece el tiempo de cada operación: el exponente $k$ de $t(32\,000) / t(1000) = 32^k$ vale cerca de 1 si el coste es lineal y cerca de 0 si es constante o logarítmico. Imprimir solo la clasificación, no los tiempos, que cambian de un ordenador a otro. Para no tardar en preparar los datos, la medición construye la representación interna directamente (con cada elemento dos veces en la lista con duplicados).

In [6]:
COSTES = {
    "insertar": ("O(m)", "O(n)", "O(n)", "O(n)"),
    "eliminar": ("O(m)", "O(n)", "O(n)", "O(n)"),
    "pertenece": ("O(m)", "O(n)", "O(log n)", "O(1)"),
    "menor": ("O(m)", "O(n)", "O(1)", "O(n)"),
    "es_vacio": ("O(1)", "O(1)", "O(1)", "O(1)"),
}
nombres = ["con duplicados", "sin duplicados", "ordenado", "frozenset"]
print(f"{'':10}" + "".join(f"{n:>16}" for n in nombres))
for operacion, costes in COSTES.items():
    print(f"{operacion:10}" + "".join(f"{c:>16}" for c in costes))

            con duplicados  sin duplicados        ordenado       frozenset
insertar              O(m)            O(n)            O(n)            O(n)
eliminar              O(m)            O(n)            O(n)            O(n)
pertenece             O(m)            O(n)        O(log n)            O(1)
menor                 O(m)            O(n)            O(1)            O(n)
es_vacio              O(1)            O(1)            O(1)            O(1)


Insertar y eliminar cuestan $O(n)$ en todas porque, para ser persistentes, copian la representación: en una versión mutable insertar en la lista con duplicados o en un `set` costaría $O(1)$, y en la lista ordenada seguiría siendo $O(n)$ por desplazar los elementos. Las consultas sí distinguen las representaciones: la búsqueda binaria y la tabla hash hacen `pertenece` sublineal, y solo la lista ordenada tiene el menor a mano.

In [7]:
def construir_directo(impl, n):
    xs = list(range(n, 0, -1))
    match impl.__name__:
        case "ConjuntoConDuplicados":
            return impl(xs + xs)
        case "ConjuntoOrdenado":
            return impl(sorted(xs))
        case _:
            return impl(xs)


def crecimiento(impl, operacion, n1=1000, n2=32_000):
    tiempos = []
    for n in (n1, n2):
        c = construir_directo(impl, n)
        llamada = {
            "insertar": lambda: c.insertar(n // 2 + 0.5),
            "eliminar": lambda: c.eliminar(n // 2),
            "pertenece": lambda: c.pertenece(-1),
            "menor": c.menor,
        }[operacion]
        tiempos.append(min(timeit.repeat(llamada, number=20, repeat=7)))
    return math.log(tiempos[1] / tiempos[0]) / math.log(n2 / n1)


print(f"{'':10}" + "".join(f"{n:>16}" for n in nombres))
for operacion in ("insertar", "eliminar", "pertenece", "menor"):
    fila = []
    for impl, coste in zip(IMPLEMENTACIONES, COSTES[operacion]):
        k = crecimiento(impl, operacion)
        clase = "lineal" if k > 0.6 else "sublineal" if k < 0.4 else "dudoso"
        assert clase == ("sublineal" if coste in ("O(1)", "O(log n)") else "lineal"), (operacion, impl, k)
        fila.append(clase)
    print(f"{operacion:10}" + "".join(f"{c:>16}" for c in fila))

            con duplicados  sin duplicados        ordenado       frozenset


insertar            lineal          lineal          lineal          lineal


eliminar            lineal          lineal          lineal          lineal
pertenece           lineal          lineal       sublineal       sublineal
menor               lineal          lineal       sublineal          lineal


## 6. Conversiones, cardinal y conjuntos unitarios

Escribir con la interfaz `lista_a_conjunto(xs, impl)`, `conjunto_a_lista(c)` (ordenada), `cardinal(c)` y `es_unitario(c)`. Comprobarlas contra `set`, `sorted` y `len` en las cuatro implementaciones, y que `lista_a_conjunto(conjunto_a_lista(c))` devuelve `c`.

In [8]:
def lista_a_conjunto(xs, impl):
    return reduce(lambda c, x: c.insertar(x), xs, impl.vacio())


def conjunto_a_lista(c):
    return list(elementos(c))


def cardinal(c):
    return sum(1 for _ in elementos(c))


def es_unitario(c):
    return not c.es_vacio() and c.eliminar(c.menor()).es_vacio()


islas = lista_a_conjunto(["Tenerife", "La Gomera", "Tenerife", "El Hierro", "La Gomera"], ConjuntoOrdenado)
print(f"{islas}: cardinal {cardinal(islas)}, unitario {es_unitario(islas)}")
print(f"{lista_a_conjunto([7, 7, 7], ConjuntoConDuplicados)}: unitario {es_unitario(lista_a_conjunto([7, 7, 7], ConjuntoConDuplicados))}")

for impl in IMPLEMENTACIONES:

    @given(st.lists(st.integers(0, 15), max_size=12))
    def conversiones(xs):
        c = lista_a_conjunto(xs, impl)
        assert conjunto_a_lista(c) == sorted(set(xs))
        assert cardinal(c) == len(set(xs))
        assert es_unitario(c) == (len(set(xs)) == 1)
        assert lista_a_conjunto(conjunto_a_lista(c), impl) == c

    assert comprobar(conversiones, f"conversiones y cardinal ({impl.__name__})") is None

{El Hierro, La Gomera, Tenerife}: cardinal 3, unitario False
{7}: unitario True
conversiones y cardinal (ConjuntoConDuplicados): pasa
conversiones y cardinal (ConjuntoSinDuplicados): pasa
conversiones y cardinal (ConjuntoOrdenado): pasa
conversiones y cardinal (ConjuntoFrozenset): pasa


## 7. Subconjuntos y conjuntos disjuntos

Escribir `es_subconjunto(a, b)`, `es_subconjunto_propio(a, b)` (subconjunto y distinto) y `son_disjuntos(a, b)`, con `a` y `b` de cualquier implementación. Comprobarlas contra `<=`, `<` e `isdisjoint` de `set`.

In [9]:
def es_subconjunto(a, b):
    return all(b.pertenece(x) for x in elementos(a))


def es_subconjunto_propio(a, b):
    return es_subconjunto(a, b) and not es_subconjunto(b, a)


def son_disjuntos(a, b):
    return not any(b.pertenece(x) for x in elementos(a))


def como_set(c):
    return set(elementos(c))


surf = lista_a_conjunto(["Ana", "Iballa", "Yeray"], ConjuntoSinDuplicados)
vela = lista_a_conjunto(["Ana", "Iballa", "Yeray", "Tanausú"], ConjuntoFrozenset)
buceo = lista_a_conjunto(["Dácil", "Elio"], ConjuntoOrdenado)
print(f"surf {surf} ⊆ vela {vela}: {es_subconjunto(surf, vela)}, propio: {es_subconjunto_propio(surf, vela)}")
print(f"vela ⊆ vela: {es_subconjunto(vela, vela)}, propio: {es_subconjunto_propio(vela, vela)}")
print(f"surf y buceo {buceo} disjuntos: {son_disjuntos(surf, buceo)}")

for impl, otra in zip(IMPLEMENTACIONES, IMPLEMENTACIONES[1:] + IMPLEMENTACIONES[:1]):

    @given(conjuntos(impl, st.integers(0, 6), 6), conjuntos(otra, st.integers(0, 6), 10))
    def relaciones(a, b):
        sa, sb = como_set(a), como_set(b)
        assert es_subconjunto(a, b) == (sa <= sb)
        assert es_subconjunto_propio(a, b) == (sa < sb)
        assert son_disjuntos(a, b) == sa.isdisjoint(sb)

    assert comprobar(relaciones, f"relaciones ({impl.__name__} con {otra.__name__})") is None

surf {Ana, Iballa, Yeray} ⊆ vela {Ana, Iballa, Tanausú, Yeray}: True, propio: True
vela ⊆ vela: True, propio: False
surf y buceo {Dácil, Elio} disjuntos: True


relaciones (ConjuntoConDuplicados con ConjuntoSinDuplicados): pasa
relaciones (ConjuntoSinDuplicados con ConjuntoOrdenado): pasa


relaciones (ConjuntoOrdenado con ConjuntoFrozenset): pasa


relaciones (ConjuntoFrozenset con ConjuntoConDuplicados): pasa


## 8. Unión e intersección

Escribir `union(a, b)` (insertando en `a` los elementos de `b`), `union_general(cs, impl)` de una lista de conjuntos, `interseccion(a, b)` e `interseccion_general(cs)`, que lanza `ValueError` con una lista vacía porque la intersección de ningún conjunto sería el universo. El resultado es de la implementación del primer argumento. Aplicarlas a las islas que ha visitado cada miembro de una familia y comprobarlas contra `|` y `&`.

In [10]:
def union(a, b):
    return reduce(lambda c, x: c.insertar(x), elementos(b), a)


def union_general(cs, impl):
    return reduce(union, cs, impl.vacio())


def interseccion(a, b):
    return reduce(lambda c, x: c.insertar(x) if b.pertenece(x) else c, elementos(a), a.vacio())


def interseccion_general(cs):
    if not cs:
        raise ValueError("la intersección de ningún conjunto no está definida sin un universo")
    return reduce(interseccion, cs)


visitadas = {
    "Carmen": lista_a_conjunto(["Tenerife", "La Palma", "Lanzarote", "Gran Canaria"], ConjuntoOrdenado),
    "Pedro": lista_a_conjunto(["Gran Canaria", "Tenerife", "Fuerteventura"], ConjuntoFrozenset),
    "Nayra": lista_a_conjunto(["Tenerife", "Gran Canaria", "El Hierro", "Tenerife"], ConjuntoConDuplicados),
}
print("Alguien ha ido :", union_general(visitadas.values(), ConjuntoOrdenado))
print("Han ido todos  :", interseccion_general(list(visitadas.values())))
print("Carmen y Pedro :", interseccion(visitadas["Carmen"], visitadas["Pedro"]))
try:
    interseccion_general([])
except ValueError as e:
    print("ValueError:", e)

for impl, otra in zip(IMPLEMENTACIONES, IMPLEMENTACIONES[1:] + IMPLEMENTACIONES[:1]):

    @given(st.lists(conjuntos(impl, max_size=8), min_size=1, max_size=4), conjuntos(otra, max_size=8))
    def union_e_interseccion(cs, b):
        a, sets = cs[0], [como_set(c) for c in cs]
        assert como_set(union(a, b)) == sets[0] | como_set(b)
        assert como_set(interseccion(a, b)) == sets[0] & como_set(b)
        assert como_set(union_general(cs, otra)) == set().union(*sets)
        assert como_set(interseccion_general(cs)) == set.intersection(*sets)

    assert comprobar(union_e_interseccion, f"unión e intersección ({impl.__name__} con {otra.__name__})") is None

Alguien ha ido : {El Hierro, Fuerteventura, Gran Canaria, La Palma, Lanzarote, Tenerife}
Han ido todos  : {Gran Canaria, Tenerife}
Carmen y Pedro : {Gran Canaria, Tenerife}
ValueError: la intersección de ningún conjunto no está definida sin un universo


unión e intersección (ConjuntoConDuplicados con ConjuntoSinDuplicados): pasa


unión e intersección (ConjuntoSinDuplicados con ConjuntoOrdenado): pasa


unión e intersección (ConjuntoOrdenado con ConjuntoFrozenset): pasa


unión e intersección (ConjuntoFrozenset con ConjuntoConDuplicados): pasa


## 9. Diferencia y diferencia simétrica

Escribir `diferencia(a, b)`, con los elementos de `a` que no están en `b`, y `diferencia_simetrica(a, b)`, con los que están en uno solo de los dos. Comprobarlas contra `-` y `^`.

In [11]:
def diferencia(a, b):
    return reduce(lambda c, x: c.eliminar(x), elementos(b), a)


def diferencia_simetrica(a, b):
    return union(diferencia(a, b), diferencia(b, a))


carmen, pedro = visitadas["Carmen"], visitadas["Pedro"]
print("Carmen sin Pedro:", diferencia(carmen, pedro))
print("Solo uno de los dos:", diferencia_simetrica(carmen, pedro))

for impl, otra in zip(IMPLEMENTACIONES, IMPLEMENTACIONES[1:] + IMPLEMENTACIONES[:1]):

    @given(conjuntos(impl), conjuntos(otra))
    def diferencias(a, b):
        assert como_set(diferencia(a, b)) == como_set(a) - como_set(b)
        assert como_set(diferencia_simetrica(a, b)) == como_set(a) ^ como_set(b)

    assert comprobar(diferencias, f"diferencias ({impl.__name__} con {otra.__name__})") is None

Carmen sin Pedro: {La Palma, Lanzarote}
Solo uno de los dos: {Fuerteventura, La Palma, Lanzarote}


diferencias (ConjuntoConDuplicados con ConjuntoSinDuplicados): pasa


diferencias (ConjuntoSinDuplicados con ConjuntoOrdenado): pasa


diferencias (ConjuntoOrdenado con ConjuntoFrozenset): pasa


diferencias (ConjuntoFrozenset con ConjuntoConDuplicados): pasa


## 10. Filtrar, partir y dividir según un pivote

Escribir `filtrar(pred, c)`; `particion(pred, c)`, que devuelva el par (los que cumplen `pred`, los que no); y `dividir(x, c)`, el par (los menores o iguales que `x`, los mayores). Comprobar que las dos partes son disjuntas, que su unión es `c` y que coinciden con comprensiones de conjuntos.

In [12]:
def filtrar(pred, c):
    return reduce(lambda r, x: r.insertar(x) if pred(x) else r, elementos(c), c.vacio())


def particion(pred, c):
    return filtrar(pred, c), filtrar(lambda x: not pred(x), c)


def dividir(x, c):
    return particion(lambda y: y <= x, c)


c = lista_a_conjunto([12, 5, 8, 21, 3, 17, 9], ConjuntoSinDuplicados)
print("c               :", c)
print("múltiplos de 3  :", filtrar(lambda x: x % 3 == 0, c))
print("pares / impares :", *particion(lambda x: x % 2 == 0, c))
print("≤ 9 / > 9       :", *dividir(9, c))

for impl in IMPLEMENTACIONES:

    @given(conjuntos(impl), st.integers(0, 15))
    def partes(c, x):
        s = como_set(c)
        si, no = particion(lambda y: y % 3 == 1, c)
        menores, mayores = dividir(x, c)
        assert como_set(si) == {y for y in s if y % 3 == 1} and como_set(no) == s - como_set(si)
        assert como_set(menores) == {y for y in s if y <= x} and como_set(mayores) == {y for y in s if y > x}
        for p, q in ((si, no), (menores, mayores)):
            assert son_disjuntos(p, q) and union(p, q) == c

    assert comprobar(partes, f"filtrar, partir y dividir ({impl.__name__})") is None

c               : {3, 5, 8, 9, 12, 17, 21}
múltiplos de 3  : {3, 9, 12, 21}
pares / impares : {8, 12} {3, 5, 9, 17, 21}
≤ 9 / > 9       : {3, 5, 8, 9} {12, 17, 21}
filtrar, partir y dividir (ConjuntoConDuplicados): pasa
filtrar, partir y dividir (ConjuntoSinDuplicados): pasa


filtrar, partir y dividir (ConjuntoOrdenado): pasa
filtrar, partir y dividir (ConjuntoFrozenset): pasa


## 11. Imagen por una función, todos y alguno

Escribir `imagen(f, c)`, el conjunto de los $f(x)$ con $x \in c$ (puede tener menos elementos que `c`), y `todos_cumplen(pred, c)` y `alguno_cumple(pred, c)`. Calcular los restos módulo 4 de un conjunto de números y comprobarlas contra comprensiones, `all` y `any`.

In [13]:
def imagen(f, c):
    return reduce(lambda r, x: r.insertar(f(x)), elementos(c), c.vacio())


def todos_cumplen(pred, c):
    return all(pred(x) for x in elementos(c))


def alguno_cumple(pred, c):
    return any(pred(x) for x in elementos(c))


c = lista_a_conjunto([3, 8, 11, 14, 20, 27], ConjuntoFrozenset)
print(f"{c} módulo 4: {imagen(lambda x: x % 4, c)}")
print("¿Todos mayores que 2?", todos_cumplen(lambda x: x > 2, c), " ¿Alguno múltiplo de 9?", alguno_cumple(lambda x: x % 9 == 0, c))

for impl in IMPLEMENTACIONES:

    @given(conjuntos(impl))
    def imagen_todos_alguno(c):
        s = como_set(c)
        assert como_set(imagen(lambda x: x // 3, c)) == {x // 3 for x in s}
        assert cardinal(imagen(lambda x: x // 3, c)) <= cardinal(c)
        assert todos_cumplen(lambda x: x < 12, c) == all(x < 12 for x in s)
        assert alguno_cumple(lambda x: x % 5 == 0, c) == any(x % 5 == 0 for x in s)

    assert comprobar(imagen_todos_alguno, f"imagen, todos y alguno ({impl.__name__})") is None

{3, 8, 11, 14, 20, 27} módulo 4: {0, 2, 3}
¿Todos mayores que 2? True  ¿Alguno múltiplo de 9? True
imagen, todos y alguno (ConjuntoConDuplicados): pasa


imagen, todos y alguno (ConjuntoSinDuplicados): pasa


imagen, todos y alguno (ConjuntoOrdenado): pasa
imagen, todos y alguno (ConjuntoFrozenset): pasa


## 12. Producto cartesiano

Escribir `producto(a, b)`, el conjunto de los pares `(x, y)` con $x \in a$ e $y \in b$, de la implementación de `a`. Formar las combinaciones de un menú con primeros y postres y comprobar que $|A \times B| = |A| \cdot |B|$ y que coincide con `itertools.product`.

In [14]:
def producto(a, b):
    return reduce(lambda r, x: reduce(lambda r, y: r.insertar((x, y)), elementos(b), r), elementos(a), a.vacio())


primeros = lista_a_conjunto(["potaje", "sancocho"], ConjuntoOrdenado)
postres = lista_a_conjunto(["bienmesabe", "frangollo", "quesillo"], ConjuntoSinDuplicados)
for plato, postre in elementos(producto(primeros, postres)):
    print(f"  {plato} + {postre}")

for impl, otra in zip(IMPLEMENTACIONES, IMPLEMENTACIONES[1:] + IMPLEMENTACIONES[:1]):

    @given(conjuntos(impl, max_size=6), conjuntos(otra, max_size=6))
    def producto_cartesiano(a, b):
        p = producto(a, b)
        assert cardinal(p) == cardinal(a) * cardinal(b)
        assert como_set(p) == set(producto_itertools(como_set(a), como_set(b)))

    assert comprobar(producto_cartesiano, f"producto ({impl.__name__} con {otra.__name__})") is None

  potaje + bienmesabe
  potaje + frangollo
  potaje + quesillo
  sancocho + bienmesabe
  sancocho + frangollo
  sancocho + quesillo
producto (ConjuntoConDuplicados con ConjuntoSinDuplicados): pasa


producto (ConjuntoSinDuplicados con ConjuntoOrdenado): pasa


producto (ConjuntoOrdenado con ConjuntoFrozenset): pasa
producto (ConjuntoFrozenset con ConjuntoConDuplicados): pasa


## 13. Leyes del álgebra de conjuntos

Comprobar con Hypothesis, en las cuatro implementaciones, las leyes de De Morgan relativas a un universo $U$ ($U - (A \cup B) = (U - A) \cap (U - B)$ y $U - (A \cap B) = (U - A) \cup (U - B)$), las distributivas ($A \cap (B \cup C) = (A \cap B) \cup (A \cap C)$ y $A \cup (B \cap C) = (A \cup B) \cap (A \cup C)$), la de absorción ($A \cup (A \cap B) = A$), $A \mathbin{\triangle} B = (A \cup B) - (A \cap B)$ y $A \subseteq B \iff A \cup B = B$. Las igualdades se comparan con el `__eq__` del TAD y, además, cada lado se contrasta con el `set` de Python.

In [15]:
for impl in IMPLEMENTACIONES:
    cs = conjuntos(impl, max_size=10)

    @given(cs, cs, cs, cs)
    def leyes(u, a, b, c):
        su, sa, sb, sc = map(como_set, (u, a, b, c))
        lado = diferencia(u, union(a, b))
        assert lado == interseccion(diferencia(u, a), diferencia(u, b)) and como_set(lado) == su - (sa | sb)
        lado = diferencia(u, interseccion(a, b))
        assert lado == union(diferencia(u, a), diferencia(u, b)) and como_set(lado) == su - (sa & sb)
        lado = interseccion(a, union(b, c))
        assert lado == union(interseccion(a, b), interseccion(a, c)) and como_set(lado) == sa & (sb | sc)
        lado = union(a, interseccion(b, c))
        assert lado == interseccion(union(a, b), union(a, c)) and como_set(lado) == sa | (sb & sc)
        assert union(a, interseccion(a, b)) == a
        lado = diferencia_simetrica(a, b)
        assert lado == diferencia(union(a, b), interseccion(a, b)) and como_set(lado) == sa ^ sb
        assert es_subconjunto(a, b) == (union(a, b) == b)

    assert comprobar(leyes, f"leyes del álgebra de conjuntos ({impl.__name__})") is None

leyes del álgebra de conjuntos (ConjuntoConDuplicados): pasa
leyes del álgebra de conjuntos (ConjuntoSinDuplicados): pasa


leyes del álgebra de conjuntos (ConjuntoOrdenado): pasa
leyes del álgebra de conjuntos (ConjuntoFrozenset): pasa


## 14. Las cuatro implementaciones coinciden

Aplicar a la vez a las cuatro implementaciones y a un `set` de Python secuencias aleatorias de inserciones y eliminaciones generadas con Hypothesis; después de cada paso deben coincidir con el `set` en `es_vacio`, en el menor y en sus elementos, y una consulta `pertenece` al azar debe dar lo mismo en todas. Añadir una implementación con un fallo, `ConjuntoConDuplicadosMal`, cuyo `eliminar` quita solo la primera aparición, y ver el contraejemplo mínimo que encuentra Hypothesis.

In [16]:
def coinciden(implementaciones):
    @given(st.lists(st.tuples(st.booleans(), st.integers(0, 9), st.integers(0, 9)), max_size=40))
    def prueba(ops):
        cs, modelo = [impl.vacio() for impl in implementaciones], set()
        for inserta, x, consulta in ops:
            if inserta:
                cs = [c.insertar(x) for c in cs]
                modelo.add(x)
            else:
                cs = [c.eliminar(x) for c in cs]
                modelo.discard(x)
            esperado = (not modelo, min(modelo, default=None), sorted(modelo), consulta in modelo)
            for c in cs:
                observado = (c.es_vacio(), None if c.es_vacio() else c.menor(), conjunto_a_lista(c), c.pertenece(consulta))
                assert observado == esperado

    return prueba


assert comprobar(coinciden(IMPLEMENTACIONES), "las cuatro implementaciones") is None


class ConjuntoConDuplicadosMal(ConjuntoConDuplicados):
    __slots__ = ()

    def eliminar(self, x):
        if x not in self._elementos:
            return self
        xs = self._elementos.copy()
        xs.remove(x)
        return type(self)(xs)


error = comprobar(coinciden(IMPLEMENTACIONES + (ConjuntoConDuplicadosMal,)), "con ConjuntoConDuplicadosMal")
assert error is not None and "(True, 0, 0)" in str(error.__notes__)
print("Basta insertar dos veces el 0: elementos() elimina el menor tras darlo, queda la otra copia y el 0 sale dos veces.")

las cuatro implementaciones: pasa
con ConjuntoConDuplicadosMal: FALLA (AssertionError)
    Failing test case: prueba(
        ops=[(True, 0, 0), (
             True,
             0,
             0,  # or any other generated value
         )],
    )
Basta insertar dos veces el 0: elementos() elimina el menor tras darlo, queda la otra copia y el 0 sale dos veces.
